# ML-эксперименты E1–E6 — дипломная работа

Пошаговый прогон шести экспериментов поверх модели v3.1 (переобученная на воспроизводимом split'е).

| # | Эксперимент | Требует весов | Требует GPU | Время |
|---|---|---|---|---|
| Retrain v3.1 | Воспроизводимое переобучение | (init: v3) | **обязательно** | ~30 мин T4 |
| E3 | Score-card per-class | нет | нет | ~5 мин |
| E1 | Grad-CAM | да | желательно | ~5 мин CPU / ~1 мин GPU |
| E2 | Embeddings + t-SNE/UMAP | да | желательно | ~5 мин |
| E4 | Калибровка вероятностей | да | нет | ~3 мин |
| E5 | CLIP linear-probe + zero-shot | нет (CLIP качается) | желательно | ~10 мин GPU |
| E6 | Multi-label fine-tune | warm-start от v3.1 | **обязательно** | ~30 мин T4 |

**Включить GPU:** Runtime → Change runtime type → **T4 GPU**.

## Почему переобучаем

Оригинальная v3 (val_acc=0.726 в `MyDrive/training_out_v3/`) обучалась на split'е, который жил в эфемерной `/content/dataset_split/` и был стёрт после ребута runtime. На любом другом split'е (даже с тем же `seed=42` — порядок `iterdir()` Drive нестабилен) часть фото попадает из train в val — это data leakage, точность инфлируется до ~0.92.

Решение — заново обучить **на воспроизводимом split'е** (тот же seed + dedup + cap + minimalist cap), сохранить веса как `v3.1`, и все экспериметы E1–E5 гонять уже на них. Тогда числа в дипломе согласованы и честные.

## Что должно лежать в Drive

```
MyDrive/
├── dataset/                              ← плоская структура по классам
└── training_out_v3/
    └── efficientnet_b0_styles.pth        ← старые веса (используются как init)
```

После прогона добавится:
```
MyDrive/
├── training_out_v3.1/                    ← новые честные веса
│   └── efficientnet_b0_styles.pth
└── diploma_out/                          ← артефакты экспериментов
```

## Шаг 0. Setup — один раз в начале сессии

In [ ]:
# 1) Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Клонируем ветку с экспериментами
!rm -rf /content/visual-style-classifier
!git clone -b ml-deep-dive \
    https://github.com/tvrvs91/visual-style-classifier.git /content/visual-style-classifier
%cd /content/visual-style-classifier

In [ ]:
# 3) Доп. зависимости
!pip install -q -r training/requirements-experiments.txt

In [ ]:
# 4) Переменные путей
import os
DRIVE = '/content/drive/MyDrive'
WEIGHTS_V3 = f'{DRIVE}/training_out_v3/efficientnet_b0_styles.pth'
WEIGHTS = f'{DRIVE}/training_out_v3.1/efficientnet_b0_styles.pth'   # сюда сохранятся новые
DATA = '/content/dataset'
OUT = f'{DRIVE}/diploma_out'
RETRAIN_OUT = f'{DRIVE}/training_out_v3.1'
os.makedirs(OUT, exist_ok=True)
os.makedirs(RETRAIN_OUT, exist_ok=True)

assert os.path.exists(f'{DRIVE}/dataset'), f'нет датасета: {DRIVE}/dataset'
print('Setup OK')
print(f'  dataset       : {DRIVE}/dataset')
print(f'  v3 (init)     : {WEIGHTS_V3} {"✓" if os.path.exists(WEIGHTS_V3) else "⚠ нет — обучим с нуля"}')
print(f'  v3.1 (target) : {WEIGHTS}')
print(f'  out           : {OUT}')

In [ ]:
# 5) Воспроизводимый v3 split (MD5-dedup + cap 600 + minimalist 500 + 80/20 seed=42)
#    Drive не меняется — split в /content/dataset/ через симлинки.
!python training/make_split.py \
    --src "$DRIVE/dataset" \
    --dst {DATA} \
    --val-ratio 0.20 --seed 42 \
    --dedup \
    --target-count 600 \
    --cap-class minimalist:500

## Шаг 1. Переобучение v3 → v3.1 (на воспроизводимом split'е)

**Те же гиперпараметры что в оригинальном `train_pipeline.ipynb`:**
- arch: EfficientNet-B0
- 5 epochs head-only (lr=1e-3) + 25 epochs full fine-tune (lr_head=1e-3, lr_backbone=1e-4)
- CosineAnnealingLR, label_smoothing=0.1, WeightedRandomSampler
- batch_size=32, mixed precision, early stopping patience=7

**Ожидание:** val_acc 0.71–0.74 (как в оригинале), но теперь честно — без data leakage.

**Время:** ~30 минут на T4. Не закрывай вкладку.

**Результат** сохранится в `MyDrive/training_out_v3.1/efficientnet_b0_styles.pth` — он автоматически подхватится во всех следующих экспериментах через переменную `WEIGHTS`.

In [ ]:
# Проверка GPU перед обучением
import torch
assert torch.cuda.is_available(), 'GPU не подключён — Runtime → Change runtime type → T4'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!python training/train.py \
    --data-dir {DATA} \
    --out-dir {RETRAIN_OUT} \
    --arch efficientnet_b0 \
    --epochs 30 \
    --head-epochs 5 \
    --batch-size 32 \
    --label-smoothing 0.1 \
    --patience 7

In [ ]:
# Проверяем что новые веса есть
assert os.path.exists(WEIGHTS), f'веса v3.1 не появились по пути {WEIGHTS} — посмотри логи train.py выше'
print(f'✓ v3.1 веса готовы: {WEIGHTS}')
print(f'  размер: {os.path.getsize(WEIGHTS) / 1e6:.1f} MB')

## Шаг 2. Sanity-check на новых весах

Прогоняем diagnose на v3.1 + том же split'е. Ожидаем acc 0.71–0.74. Если так — все последующие эксперименты пойдут на честных метриках.

In [ ]:
!python training/diagnose.py \
    --weights {WEIGHTS} \
    --val-dir {DATA}/val \
    --arch efficientnet_b0 \
    --out-dir {OUT}/diagnose_v3.1

## E3. Score-card per-class анализ

Самый быстрый, GPU не нужен. Не использует веса — только датасет.

**Что получаем:** `radar.png`, `violins.png`, `stats.csv`, `anova.json`, `baseline.json`

In [ ]:
!python training/score_card_class.py \
    --data-dir {DATA} \
    --out-dir {OUT}/scorecard \
    --splits train val

In [ ]:
from IPython.display import Image, display
import json
display(Image(f'{OUT}/scorecard/radar.png'))
display(Image(f'{OUT}/scorecard/violins.png'))
print(json.dumps(json.load(open(f'{OUT}/scorecard/baseline.json')), indent=2, ensure_ascii=False)[:1500])

## E1. Grad-CAM визуализация attention

**Что получаем:** `gradcam_grid.png`, `gradcam_<class>.png`, `gradcam_errors.png`, `notes.json`

In [ ]:
!python training/gradcam_viz.py \
    --weights {WEIGHTS} \
    --data-dir {DATA} \
    --out-dir {OUT}/gradcam \
    --per-class 6 --errors 10

In [ ]:
display(Image(f'{OUT}/gradcam/gradcam_grid.png'))
display(Image(f'{OUT}/gradcam/gradcam_errors.png'))

## E2. Embeddings + t-SNE / UMAP

**Что получаем:** `tsne.png`, `umap.png`, `metrics.json`, `centroid_similarity.png/.csv`, `embeddings.npy`, `labels.npy`

In [ ]:
!python training/embed_viz.py \
    --weights {WEIGHTS} \
    --data-dir {DATA} \
    --out-dir {OUT}/embed \
    --split val --umap

In [ ]:
display(Image(f'{OUT}/embed/tsne.png'))
display(Image(f'{OUT}/embed/umap.png'))
display(Image(f'{OUT}/embed/centroid_similarity.png'))
print(json.dumps(json.load(open(f'{OUT}/embed/metrics.json')), indent=2, ensure_ascii=False))

## E4. Калибровка вероятностей

Метод Guo et al. 2017. **Что получаем:** `reliability_combined.png`, `metrics.json`, `temperature.txt`

In [ ]:
!python training/calibration.py \
    --weights {WEIGHTS} \
    --data-dir {DATA} \
    --out-dir {OUT}/calibration \
    --vector-scaling

In [ ]:
display(Image(f'{OUT}/calibration/reliability_combined.png'))
print('Найденное T:', open(f'{OUT}/calibration/temperature.txt').read())
print(json.dumps(json.load(open(f'{OUT}/calibration/metrics.json')), indent=2, ensure_ascii=False))

## E5. CLIP linear-probe + zero-shot

Качает ~350 МБ модели. **Что получаем:** `comparison_table.csv`, `confusion_matrix_lp/zs.png`, `tsne_clip.png`

In [ ]:
!python training/clip_probe.py \
    --data-dir {DATA} \
    --out-dir {OUT}/clip \
    --model ViT-B-32 --pretrained openai

In [ ]:
import csv
with open(f'{OUT}/clip/comparison_table.csv') as f:
    for row in csv.reader(f):
        print(row)
display(Image(f'{OUT}/clip/confusion_matrix_lp.png'))
display(Image(f'{OUT}/clip/tsne_clip.png'))

## E6. Multi-label fine-tune (опционально, требует GPU)

Тёплый старт от v3.1. ~30 мин на T4.

**Что получаем:** `efficientnet_b0_multilabel.pth`, `thresholds.json`, `metrics.json`, `co_label_stats.json`

In [ ]:
!python training/train_multilabel.py \
    --data-dir {DATA} \
    --out-dir {OUT}/multilabel \
    --init-weights {WEIGHTS} \
    --epochs 25 --head-epochs 5 \
    --batch-size 32 --num-workers 2 \
    --patience 7

In [ ]:
metrics = json.load(open(f'{OUT}/multilabel/metrics.json'))
print('mAP:', metrics['mAP'])
print('macro F1:', metrics['macro_f1'])
print('exact_match:', metrics['exact_match_accuracy'])
print('hamming:', metrics['hamming_accuracy'])
print()
print('Per-class:')
for cls, m in metrics['per_class'].items():
    print(f"  {cls:14s}  F1={m['f1']:.3f}  AP={m['AP']:.3f}  thr={m['threshold']:.2f}")

## Финальная сводка

После прохода всех ячеек в Drive будет:

```
MyDrive/
├── training_out_v3/      (старая, инфлированная — оставляем как baseline для сравнения)
├── training_out_v3.1/    (НОВАЯ, честная — это финальная модель)
└── diploma_out/
    ├── diagnose_v3.1/    (sanity-check на новой модели)
    ├── scorecard/        (E3)
    ├── gradcam/          (E1)
    ├── embed/            (E2)
    ├── calibration/      (E4)
    ├── clip/             (E5)
    └── multilabel/       (E6, опционально)
```

После завершения пришли мне путь — я переведу артефакты в текст для дипломного отчёта.